In [21]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.utils import compute_class_weight

# 1. Data Loading and Preprocessing
def load_and_preprocess():
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Feature engineering
    data['date'] = pd.to_datetime(data['date'])
    data['rest_to_eat_ratio'] = np.where(
        (data['EAT'] == 0) & (data['REST'] == 0),
        1.0,
        data['REST'] / (data['EAT'] + 1e-6)
    )
    
    # Cyclical time features
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    return data[features], data[targets], data['cow']

# 2. Custom Class Weight Calculation
def calculate_class_weights(y):
    class_weights = []
    for i, target in enumerate(y.columns):
        classes = np.unique(y.iloc[:,i])
        weights = compute_class_weight('balanced', classes=classes, y=y.iloc[:,i])
        class_weights.append(dict(zip(classes, weights)))
    return class_weights

# 3. MLP Model Definition with Class Weights
def create_mlp_model():
    # Create base MLP model
    base_mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=0.001,
        batch_size=512,
        learning_rate='adaptive',
        early_stopping=True,
        max_iter=1000,
        random_state=42
    )
    
    # Create pipeline
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('mlp', MultiOutputClassifier(base_mlp))
    ])
    
    return model

# 4. Training with GroupKFold and Class Weights
def train_mlp_model():
    X, y, groups = load_and_preprocess()
    class_weights = calculate_class_weights(y)
    model = create_mlp_model()
    
    gkf = GroupKFold(n_splits=3)
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        print(f"\nTraining Fold {fold+1}...")
        
        # Apply class weights by modifying the input data
        # Duplicate minority class samples to balance classes
        max_count = max(y_train.sum(axis=0))
        resampled_X = []
        resampled_y = []
        
        for i, target in enumerate(y.columns):
            minority_indices = y_train[y_train[target] == 1].index
            repeat_times = int(max_count / len(minority_indices)) - 1
            if repeat_times > 0:
                resampled_X.append(pd.concat([X_train.loc[minority_indices]] * repeat_times))
                resampled_y.append(pd.concat([y_train.loc[minority_indices]] * repeat_times))
        
        if resampled_X:
            X_train = pd.concat([X_train] + resampled_X)
            y_train = pd.concat([y_train] + resampled_y)
        
        model.fit(X_train, y_train)
        
        # Predict with adjusted thresholds
        y_proba = model.predict_proba(X_test)
        thresholds = [0.3, 0.2, 0.4, 0.1]  # Disease-specific thresholds
        y_pred = np.array([(y_proba[i][:,1] > thresholds[i]).astype(int) for i in range(4)]).T
        
        for i, target in enumerate(y.columns):
            print(f"\n{target} Classification Report:")
            print(classification_report(y_test.iloc[:,i], y_pred[:,i]))
    
    joblib.dump(model, 'mlp_cattle_model.joblib')
    print("\nMLP model training complete and saved.")

# 5. Prediction Function
def predict_with_mlp(new_data):
    model = joblib.load('mlp_cattle_model.joblib')
    
    # Feature engineering
    new_data = new_data.copy()
    new_data['rest_to_eat_ratio'] = np.where(
        (new_data['EAT'] == 0) & (new_data['REST'] == 0),
        1.0,
        new_data['REST'] / (new_data['EAT'] + 1e-6)
    )
    new_data['hour_sin'] = np.sin(2 * np.pi * new_data['hour'] / 24)
    new_data['hour_cos'] = np.cos(2 * np.pi * new_data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    X_new = new_data[features]
    
    # Get probabilities
    y_proba = model.predict_proba(X_new)
    thresholds = [0.3, 0.2, 0.4, 0.1]
    results = pd.DataFrame({
        'oestrus': (y_proba[0][:,1] > thresholds[0]).astype(int),
        'oestrus_prob': y_proba[0][:,1],
        'calving': (y_proba[1][:,1] > thresholds[1]).astype(int),
        'calving_prob': y_proba[1][:,1],
        'lameness': (y_proba[2][:,1] > thresholds[2]).astype(int),
        'lameness_prob': y_proba[2][:,1],
        'mastitis': (y_proba[3][:,1] > thresholds[3]).astype(int),
        'mastitis_prob': y_proba[3][:,1]
    })
    
    return results

# Example usage
if __name__ == "__main__":
    # Train the model
    train_mlp_model()
    
    # Example prediction
    new_data = pd.DataFrame({
        'IN_ALLEYS': [35.6, 0.0, 10.2],
        'REST': [3564.4, 3599.9, 2500.0],
        'EAT': [0.0, 0.0, 500.0],
        'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
        'hour': [1, 2, 3],
        'date': ['2023-01-01', '2023-01-02', '2023-01-03']
    })
    
    predictions = predict_with_mlp(new_data)
    print("\nMLP Predictions:")
    print(predictions)


Training Fold 1...

oestrus Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    642649
           1       0.00      0.00      0.00      2544

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       0.99      1.00      0.99    645193


calving Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    643777
           1       0.00      0.00      0.00      1416

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       1.00      1.00      1.00    645193


lameness Classification Report:


...

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    643652
           1       0.00      0.00      0.00      1512

    accuracy                           1.00    645164
   macro avg       0.50      0.50      0.50    645164
weighted avg       1.00      1.00      1.00    645164


lameness Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    644228
           1       0.00      0.00      0.00       936

    accuracy                           1.00    645164
   macro avg       0.50      0.50      0.50    645164
weighted avg       1.00      1.00      1.00    645164


mastitis Classification Report:


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    644636
           1       0.00      0.00      0.00       528

    accuracy                           1.00    645164
   macro avg       0.50      0.50      0.50    645164
weighted avg       1.00      1.00      1.00    645164


MLP model training complete and saved.

MLP Predictions:
   oestrus  oestrus_prob  calving  calving_prob  lameness  lameness_prob  \
0        0      0.002227        0      0.000359         0       0.002711   
1        0      0.001915        0      0.000331         0       0.002603   
2        0      0.002906        0      0.000977         0       0.002645   

   mastitis  mastitis_prob  
0         0       0.003800  
1         0       0.003848  
2         0       0.003291  


In [27]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.utils import compute_class_weight

# 1. Data Loading and Preprocessing
def load_and_preprocess():
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Feature engineering
    data['date'] = pd.to_datetime(data['date'])
    data['rest_to_eat_ratio'] = np.where(
        (data['EAT'] == 0) & (data['REST'] == 0),
        1.0,
        data['REST'] / (data['EAT'] + 1e-6)
    )
    
    # Cyclical time features
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    return data[features], data[targets], data['cow']

# 2. Custom Class Weight Calculation
def calculate_class_weights(y):
    class_weights = []
    for i, target in enumerate(y.columns):
        classes = np.unique(y.iloc[:,i])
        weights = compute_class_weight('balanced', classes=classes, y=y.iloc[:,i])
        class_weights.append(dict(zip(classes, weights)))
    return class_weights

# 3. MLP Model Definition with Class Weights
def create_mlp_model(class_weights):
    # Create base MLP model with parameters
    base_mlp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        alpha=0.001,
        batch_size=512,
        learning_rate='adaptive',
        early_stopping=True,
        max_iter=1000,
        random_state=42
    )
    
    # Create MultiOutputClassifier
    multi_output = MultiOutputClassifier(base_mlp)
    
    # Create simple pipeline with just scaler and classifier
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', multi_output)
    ])
    
    return model

# 4. Training with GroupKFold
def train_mlp_model():
    X, y, groups = load_and_preprocess()
    class_weights = calculate_class_weights(y)
    model = create_mlp_model(class_weights)
    
    gkf = GroupKFold(n_splits=3)
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        print(f"\nTraining Fold {fold+1}...")
        
        # Apply class weights by modifying the input data
        # We'll duplicate minority class samples to balance classes
        max_count = max(y_train.sum(axis=0))
        resampled_X = []
        resampled_y = []
        
        for i, target in enumerate(y.columns):
            minority_indices = y_train[y_train[target] == 1].index
            repeat_times = int(max_count / len(minority_indices)) - 1
            if repeat_times > 0:
                resampled_X.append(pd.concat([X_train.loc[minority_indices]] * repeat_times))
                resampled_y.append(pd.concat([y_train.loc[minority_indices]] * repeat_times))
        
        if resampled_X:
            X_train = pd.concat([X_train] + resampled_X)
            y_train = pd.concat([y_train] + resampled_y)
        
        model.fit(X_train, y_train)
        
        # Predict with adjusted thresholds
        y_proba = model.predict_proba(X_test)
        thresholds = [0.3, 0.2, 0.4, 0.1]  # Disease-specific thresholds
        y_pred = np.array([(y_proba[i][:,1] > thresholds[i]).astype(int) for i in range(4)]).T
        
        for i, target in enumerate(y.columns):
            print(f"\n{target} Classification Report (Threshold={thresholds[i]}):")
            print(classification_report(y_test.iloc[:,i], y_pred[:,i], zero_division=0))
    
    joblib.dump(model, 'mlp_cattle_model.joblib')
    print("\nMLP model training complete and saved.")

# 5. Prediction Function
def predict_with_mlp(new_data):
    model = joblib.load('mlp_cattle_model.joblib')
    
    # Feature engineering
    new_data = new_data.copy()
    new_data['rest_to_eat_ratio'] = np.where(
        (new_data['EAT'] == 0) & (new_data['REST'] == 0),
        1.0,
        new_data['REST'] / (new_data['EAT'] + 1e-6)
    )
    new_data['hour_sin'] = np.sin(2 * np.pi * new_data['hour'] / 24)
    new_data['hour_cos'] = np.cos(2 * np.pi * new_data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    X_new = new_data[features]
    
    # Get probabilities
    y_proba = model.predict_proba(X_new)
    thresholds = [0.3, 0.2, 0.4, 0.1]
    results = pd.DataFrame({
        'oestrus': (y_proba[0][:,1] > thresholds[0]).astype(int),
        'oestrus_prob': y_proba[0][:,1],
        'calving': (y_proba[1][:,1] > thresholds[1]).astype(int),
        'calving_prob': y_proba[1][:,1],
        'lameness': (y_proba[2][:,1] > thresholds[2]).astype(int),
        'lameness_prob': y_proba[2][:,1],
        'mastitis': (y_proba[3][:,1] > thresholds[3]).astype(int),
        'mastitis_prob': y_proba[3][:,1]
    })
    
    return results

# Example usage
if __name__ == "__main__":
    # Train the model
    train_mlp_model()
    
    # Example prediction
    new_data = pd.DataFrame({
        'IN_ALLEYS': [35.6, 0.0, 10.2],
        'REST': [3564.4, 3599.9, 2500.0],
        'EAT': [0.0, 0.0, 500.0],
        'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
        'hour': [1, 2, 3],
        'date': ['2023-01-01', '2023-01-02', '2023-01-03']
    })
    
    predictions = predict_with_mlp(new_data)
    print("\nMLP Predictions:")
    print(predictions)


Training Fold 1...

oestrus Classification Report (Threshold=0.3):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    642649
           1       0.00      0.00      0.00      2544

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       0.99      1.00      0.99    645193


calving Classification Report (Threshold=0.2):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    643777
           1       0.00      0.00      0.00      1416

    accuracy                           1.00    645193
   macro avg       0.50      0.50      0.50    645193
weighted avg       1.00      1.00      1.00    645193


lameness Classification Report (Threshold=0.4):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    644089
           1       0.00      0.00      0.00      1104

    accuracy     

In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import classification_report, f1_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.utils import compute_class_weight
from scipy.stats import loguniform
from itertools import product
from sklearn.utils.validation import check_consistent_length
from sklearn.base import clone

# 1. Enhanced Data Loading with Validation
def load_and_preprocess():
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Validate required columns exist
    required_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour', 'date', 
                    'oestrus', 'calving', 'lameness', 'mastitis', 'cow']
    missing_cols = set(required_cols) - set(data.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Feature engineering with safe division
    data['date'] = pd.to_datetime(data['date'], errors='coerce')
    
    # Convert to pandas Series before fillna
    rest_to_eat_ratio = pd.Series(
        np.where(
            (data['EAT'].values == 0) & (data['REST'].values == 0),
            1.0,
            data['REST'].values / np.where(data['EAT'].values == 0, np.nan, data['EAT'].values)
        )
    )
    data['rest_to_eat_ratio'] = rest_to_eat_ratio.fillna(1.0)
    
    # Cyclical time features
    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    # Drop rows with missing critical data
    data = data.dropna(subset=features + targets + ['cow'])
    
    return data[features], data[targets], data['cow']
# 2. Robust Class Weight Calculation
def calculate_class_weights(y):
    class_weights = []
    for i, target in enumerate(y.columns):
        classes = np.unique(y.iloc[:,i])
        weights = compute_class_weight('balanced', classes=classes, y=y.iloc[:,i])
        class_weights.append(dict(zip(classes, weights)))
    return class_weights

# 3. MLP Pipeline with Safe Configuration
def create_mlp_pipeline():
    base_mlp = MLPClassifier(
        activation='relu',
        solver='adam',
        batch_size=512,
        early_stopping=True,
        random_state=42,
        max_iter=1000
    )
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', MultiOutputClassifier(base_mlp))
    ])
    
    return pipeline

# 4. Hyperparameter Tuning with Group Validation
def tune_hyperparameters(X, y, groups):
    pipeline = create_mlp_pipeline()
    
    param_dist = {
        'classifier__estimator__hidden_layer_sizes': [(64,), (128, 64), (256, 128)],
        'classifier__estimator__alpha': loguniform(1e-4, 1e-1),
        'classifier__estimator__learning_rate_init': loguniform(1e-4, 1e-2),
        'classifier__estimator__beta_1': [0.8, 0.9, 0.95],
        'classifier__estimator__beta_2': [0.99, 0.999]
    }
    
    # Verify group consistency
    check_consistent_length(X, y, groups)
    unique_groups, group_counts = np.unique(groups, return_counts=True)
    print(f"Found {len(unique_groups)} unique groups with sizes {group_counts.min()}-{group_counts.max()}")
    
    gkf = GroupKFold(n_splits=3)
    search = RandomizedSearchCV(
        pipeline,
        param_dist,
        n_iter=15,
        cv=gkf,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=2,
        random_state=42,
        error_score='raise'
    )
    
    search.fit(X, y, groups=groups)
    print("Best parameters:", search.best_params_)
    return search.best_estimator_

# 5. Threshold Optimization with Validation
def optimize_thresholds(model, X_val, y_val):
    y_proba = [estimator.predict_proba(X_val)[:, 1] 
              for estimator in model.named_steps['classifier'].estimators_]
    
    thresholds = []
    for i in range(y_val.shape[1]):
        precision, recall, thresh = precision_recall_curve(y_val.iloc[:,i], y_proba[i])
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
        best_idx = np.argmax(f1_scores)
        thresholds.append(thresh[best_idx])
    
    return thresholds

# 6. Main Training Function with Data Integrity Checks
def train_and_evaluate():
    X, y, groups = load_and_preprocess()
    print(f"Initial data: {len(X)} samples, {len(np.unique(groups))} groups")
    
    # Ensure we have enough samples per group
    min_samples_per_group = 5
    group_counts = groups.value_counts()
    valid_groups = group_counts[group_counts >= min_samples_per_group].index
    valid_mask = groups.isin(valid_groups)
    
    X = X[valid_mask]
    y = y[valid_mask]
    groups = groups[valid_mask]
    print(f"After filtering: {len(X)} samples, {len(np.unique(groups))} groups")
    
    # Train-test split preserving groups
    unique_groups = np.unique(groups)
    np.random.seed(42)
    train_groups = np.random.choice(unique_groups, 
                                 size=int(0.8*len(unique_groups)), 
                                 replace=False)
    train_mask = groups.isin(train_groups)
    
    X_train, X_val = X[train_mask], X[~train_mask]
    y_train, y_val = y[train_mask], y[~train_mask]
    groups_train = groups[train_mask]
    
    # Class balancing
    max_count = max(y_train.sum(axis=0))
    resampled_X, resampled_y, resampled_groups = [], [], []
    
    for i in range(y.shape[1]):
        minority_indices = y_train[y_train.iloc[:,i] == 1].index
        if len(minority_indices) == 0:
            continue
            
        repeat_times = int(max_count / len(minority_indices)) - 1
        if repeat_times > 0:
            resampled_X.append(pd.concat([X_train.loc[minority_indices]] * repeat_times))
            resampled_y.append(pd.concat([y_train.loc[minority_indices]] * repeat_times))
            resampled_groups.append(pd.concat([groups_train.loc[minority_indices]] * repeat_times))
    
    if resampled_X:
        X_train = pd.concat([X_train] + resampled_X)
        y_train = pd.concat([y_train] + resampled_y)
        groups_train = pd.concat([groups_train] + resampled_groups)
    
    # Rest of the function remains the same...
    # Hyperparameter tuning
    print("\nStarting hyperparameter tuning...")
    best_model = tune_hyperparameters(X_train, y_train, groups_train)
    
    # Threshold optimization
    print("\nOptimizing prediction thresholds...")
    thresholds = optimize_thresholds(best_model, X_val, y_val)
    print(f"Optimal thresholds: {thresholds}")
    
    # Final evaluation
    y_proba = [estimator.predict_proba(X_val)[:, 1] 
              for estimator in best_model.named_steps['classifier'].estimators_]
    y_pred = np.array([(y_proba[i] > thresholds[i]).astype(int) for i in range(4)]).T
    
    print("\nValidation Set Performance:")
    for i, target in enumerate(y.columns):
        print(f"\n--- {target} (Threshold={thresholds[i]:.3f}) ---")
        print(classification_report(y_val.iloc[:,i], y_pred[:,i], zero_division=0))
    
    # Save complete model package
    model_package = {
        'model': best_model,
        'thresholds': thresholds,
        'features': X.columns.tolist(),
        'class_weights': calculate_class_weights(y),
        'timestamp': pd.Timestamp.now()
    }
    
    joblib.dump(model_package, 'cattle_health_mlp_model.pkl')
    print("\nModel training complete and saved.")
    return best_model, thresholds

# 7. Prediction Function with Input Validation
def predict_with_model(new_data):
    try:
        model_package = joblib.load('cattle_health_mlp_model.pkl')
    except FileNotFoundError:
        raise RuntimeError("Model file not found. Please train the model first.")
    
    # Input validation
    required_features = model_package['features']
    missing_features = set(required_features) - set(new_data.columns)
    if missing_features:
        raise ValueError(f"Missing required features: {missing_features}")
    
    # Feature engineering
    new_data = new_data.copy()
    if 'EAT' in new_data.columns and 'REST' in new_data.columns:
        new_data['rest_to_eat_ratio'] = np.where(
            (new_data['EAT'] == 0) & (new_data['REST'] == 0),
            1.0,
            new_data['REST'] / (new_data['EAT'].replace(0, np.nan))
        ).fillna(1.0)
    
    if 'hour' in new_data.columns:
        new_data['hour_sin'] = np.sin(2 * np.pi * new_data['hour'] / 24)
        new_data['hour_cos'] = np.cos(2 * np.pi * new_data['hour'] / 24)
    
    X_new = new_data[required_features]
    
    # Predict with optimized thresholds
    y_proba = [estimator.predict_proba(X_new)[:, 1] 
              for estimator in model_package['model'].named_steps['classifier'].estimators_]
    
    results = pd.DataFrame({
        'oestrus': (y_proba[0] > model_package['thresholds'][0]).astype(int),
        'oestrus_prob': y_proba[0],
        'calving': (y_proba[1] > model_package['thresholds'][1]).astype(int),
        'calving_prob': y_proba[1],
        'lameness': (y_proba[2] > model_package['thresholds'][2]).astype(int),
        'lameness_prob': y_proba[2],
        'mastitis': (y_proba[3] > model_package['thresholds'][3]).astype(int),
        'mastitis_prob': y_proba[3]
    })
    
    return results

if __name__ == "__main__":
    try:
        model, thresholds = train_and_evaluate()
        
        # Example prediction
        test_data = pd.DataFrame({
            'IN_ALLEYS': [35.6, 0.0, 10.2],
            'REST': [3564.4, 3599.9, 2500.0],
            'EAT': [0.0, 0.0, 500.0],
            'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
            'hour': [1, 2, 3]
        })
        
        predictions = predict_with_model(test_data)
        print("\nExample Predictions:")
        print(predictions)
        
    except Exception as e:
        print(f"Error: {str(e)}")
        raise

Initial data: 1935667 samples, 279 groups
After filtering: 1935667 samples, 279 groups

Starting hyperparameter tuning...
Found 223 unique groups with sizes 517-9720
Fitting 3 folds for each of 15 candidates, totalling 45 fits


KeyboardInterrupt: 

In [14]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import classification_report, f1_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from sklearn.utils import compute_class_weight
from scipy.stats import loguniform
import warnings
from sklearn.exceptions import ConvergenceWarning
from tqdm import tqdm

# Suppress convergence warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# 1. Optimized Data Loading
def load_and_preprocess():
    print("Loading and preprocessing data...")
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Validate required columns
    required_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour', 'date', 
                    'oestrus', 'calving', 'lameness', 'mastitis', 'cow']
    missing_cols = set(required_cols) - set(data.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Feature engineering - fixed to handle numpy arrays properly
    rest_arr = data['REST'].values
    eat_arr = data['EAT'].values
    
    # Calculate ratio safely
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(
            (eat_arr == 0) & (rest_arr == 0),
            1.0,
            rest_arr / np.where(eat_arr == 0, np.nan, eat_arr)
        )
    data['rest_to_eat_ratio'] = pd.Series(ratio).fillna(1.0)
    
    # Cyclical time features
    hour_arr = data['hour'].values
    data['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
    data['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    # Clean data
    data = data.dropna(subset=features + targets + ['cow'])
    
    return data[features], data[targets], data['cow']

# 2. Fast MLP Configuration
def create_mlp_pipeline():
    base_mlp = MLPClassifier(
        hidden_layer_sizes=(128,),
        activation='relu',
        solver='adam',
        batch_size=1024,
        learning_rate='adaptive',
        early_stopping=True,
        max_iter=200,
        tol=1e-3,
        random_state=42
    )
    
    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', MultiOutputClassifier(base_mlp, n_jobs=-1))
    ])

# 3. Efficient Hyperparameter Tuning
def tune_hyperparameters(X, y, groups):
    print("\nStarting hyperparameter tuning...")
    pipeline = create_mlp_pipeline()
    
    param_dist = {
        'classifier__estimator__hidden_layer_sizes': [(64,), (128,), (64, 32)],
        'classifier__estimator__alpha': [0.0001, 0.001, 0.01],
        'classifier__estimator__learning_rate_init': [0.001, 0.01]
    }
    
    search = RandomizedSearchCV(
        pipeline,
        param_dist,
        n_iter=5,
        cv=GroupKFold(n_splits=3),
        scoring='f1_macro',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    
    search.fit(X, y, groups=groups)
    print("Best parameters found:", search.best_params_)
    return search.best_estimator_

# 4. Threshold Optimization
def optimize_thresholds(model, X_val, y_val):
    print("\nOptimizing thresholds...")
    y_proba = [estimator.predict_proba(X_val)[:, 1] 
              for estimator in model.named_steps['classifier'].estimators_]
    
    thresholds = []
    for i in tqdm(range(y_val.shape[1]), desc="Thresholds"):
        precision, recall, thresh = precision_recall_curve(y_val.iloc[:,i], y_proba[i])
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
        thresholds.append(thresh[np.argmax(f1_scores)])
    
    return thresholds

# 5. Main Training Function
def train_and_evaluate():
    # Load data
    X, y, groups = load_and_preprocess()
    
    # Prototyping mode - use subset of data
    sample_frac = 0.3  # Use 30% of data for faster testing
    if sample_frac < 1.0:
        unique_groups = groups.unique()
        selected = np.random.choice(unique_groups, size=int(len(unique_groups)*sample_frac), replace=False)
        mask = groups.isin(selected)
        X, y, groups = X[mask], y[mask], groups[mask]
        print(f"\nUsing {sample_frac*100}% of data for prototyping ({len(X)} samples)")
    
    # Train-test split
    unique_groups = groups.unique()
    train_groups = np.random.choice(unique_groups, size=int(0.8*len(unique_groups)), replace=False)
    train_mask = groups.isin(train_groups)
    
    X_train, X_val = X[train_mask], X[~train_mask]
    y_train, y_val = y[train_mask], y[~train_mask]
    groups_train = groups[train_mask]
    
    # Train model
    model = tune_hyperparameters(X_train, y_train, groups_train)
    
    # Optimize thresholds
    thresholds = optimize_thresholds(model, X_val, y_val)
    
    # Evaluate
    y_proba = [estimator.predict_proba(X_val)[:, 1] 
              for estimator in model.named_steps['classifier'].estimators_]
    y_pred = np.array([(y_proba[i] > thresholds[i]).astype(int) for i in range(4)]).T
    
    print("\nValidation Performance:")
    for i, target in enumerate(y.columns):
        print(f"\n{target} (Threshold={thresholds[i]:.3f}):")
        print(classification_report(y_val.iloc[:,i], y_pred[:,i], zero_division=0))
    
    # Save model
    model_package = {
        'model': model,
        'thresholds': thresholds,
        'features': X.columns.tolist()
    }
    joblib.dump(model_package, 'cattle_health_model.pkl')
    print("\nModel saved successfully!")
    return model, thresholds

# 6. Prediction Function (Fixed Syntax Error)
def predict(new_data):
    try:
        model_package = joblib.load('cattle_health_model.pkl')
    except FileNotFoundError:
        raise RuntimeError("Model not found. Train the model first.")
    
    # Feature engineering
    X_new = new_data.copy()
    if {'EAT', 'REST'}.issubset(X_new.columns):
        X_new['rest_to_eat_ratio'] = np.where(
            (X_new['EAT'] == 0) & (X_new['REST'] == 0),
            1.0,
            X_new['REST'] / X_new['EAT'].replace(0, np.nan)
        ).fillna(1.0)
    
    if 'hour' in X_new.columns:
        X_new['hour_sin'] = np.sin(2 * np.pi * X_new['hour'] / 24)
        X_new['hour_cos'] = np.cos(2 * np.pi * X_new['hour'] / 24)
    
    # Corrected prediction line with proper indentation
    y_proba = [
        estimator.predict_proba(X_new[model_package['features']])[:, 1] 
        for estimator in model_package['model'].named_steps['classifier'].estimators_
    ]
    
    results = pd.DataFrame({
        'oestrus': (y_proba[0] > model_package['thresholds'][0]).astype(int),
        'oestrus_prob': y_proba[0],
        'calving': (y_proba[1] > model_package['thresholds'][1]).astype(int),
        'calving_prob': y_proba[1],
        'lameness': (y_proba[2] > model_package['thresholds'][2]).astype(int),
        'lameness_prob': y_proba[2],
        'mastitis': (y_proba[3] > model_package['thresholds'][3]).astype(int),
        'mastitis_prob': y_proba[3]
    })
    
    return results

if __name__ == "__main__":
    try:
        print("Starting cattle health prediction model training...")
        model, thresholds = train_and_evaluate()
        
        # Example prediction
        test_data = pd.DataFrame({
            'IN_ALLEYS': [35.6, 0.0, 10.2],
            'REST': [3564.4, 3599.9, 2500.0],
            'EAT': [0.0, 0.0, 500.0],
            'ACTIVITY_LEVEL': [-814.1, -827.9, -700.0],
            'hour': [1, 2, 3]
        })
        
        print("\nExample predictions:")
        print(predict(test_data))
        
    except Exception as e:
        print(f"\nError: {str(e)}")

Starting cattle health prediction model training...
Loading and preprocessing data...

Using 30.0% of data for prototyping (607162 samples)

Starting hyperparameter tuning...
Fitting 3 folds for each of 5 candidates, totalling 15 fits


C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(


Best parameters found: {'classifier__estimator__learning_rate_init': 0.001, 'classifier__estimator__hidden_layer_sizes': (64,), 'classifier__estimator__alpha': 0.0001}

Optimizing thresholds...


Thresholds: 100%|██████████| 4/4 [00:00<00:00, 103.37it/s]
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
C:\Users\vishn\anaconda3\Lib\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(



Validation Performance:

oestrus (Threshold=0.000):
              precision    recall  f1-score   support

           0       0.99      0.92      0.95    124674
           1       0.00      0.07      0.01       624

    accuracy                           0.91    125298
   macro avg       0.50      0.49      0.48    125298
weighted avg       0.99      0.91      0.95    125298


calving (Threshold=0.000):
              precision    recall  f1-score   support

           0       1.00      0.92      0.96    125082
           1       0.00      0.02      0.00       216

    accuracy                           0.92    125298
   macro avg       0.50      0.47      0.48    125298
weighted avg       1.00      0.92      0.96    125298


lameness (Threshold=0.000):
              precision    recall  f1-score   support

           0       1.00      0.93      0.96    124938
           1       0.00      0.06      0.01       360

    accuracy                           0.93    125298
   macro avg      